# Training del Modello di Machine Learning

Questo notebook si occupa della fase di **addestramento e valutazione** del sistema di triage automatico dei ticket.

Presuppone che il preprocessing descritto in `Preprocessing.ipynb` sia già stato eseguito.

I passaggi eseguiti sono:
1. Caricamento e preparazione dei dati
2. Suddivisione Training/Test (80/20)
3. Vettorizzazione TF-IDF
4. Addestramento dei modelli (Regressione Logistica)
5. Valutazione con metriche: Accuracy, Precision, Recall, F1-score
6. Visualizzazione della matrice di confusione
7. Salvataggio dei modelli per la dashboard Streamlit

## 1. Importazione delle librerie

In [ ]:
import re
import json
import pickle
import unicodedata
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

## 2. Caricamento e preparazione dei dati

In [ ]:
def clean_text(s):
    if s is None: return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s).lower()
    s = s.replace("\n", " ").replace("\t", " ")
    s = re.sub(r"[^\w\sàèéìòù]", " ", s, flags=re.UNICODE)
    return re.sub(r"\s+", " ", s).strip()

df = pd.read_csv("dataset_tickets_pw18.csv")
df["oggetto"] = df["oggetto"].fillna("").apply(clean_text)
df["descrizione"] = df["descrizione"].fillna("").apply(clean_text)
df = df.drop_duplicates(subset=["oggetto", "descrizione"]).reset_index(drop=True)
df["testo"] = (df["oggetto"] + " " + df["descrizione"]).str.strip()
print(f"Dataset pronto: {len(df)} ticket")

## 3. Suddivisione Training/Test (80/20)

`random_state=42` garantisce riproducibilità. `stratify` mantiene la proporzione delle classi in entrambi i set.

In [ ]:
X_train, X_test, y_train_cat, y_test_cat, y_train_prio, y_test_prio = train_test_split(
    df["testo"], df["categoria"], df["priorita"],
    test_size=0.2, random_state=42, stratify=df["categoria"]
)
print(f"Training set: {len(X_train)} ticket | Test set: {len(X_test)} ticket")

## 4. Vettorizzazione TF-IDF

TF-IDF trasforma il testo in numeri assegnando un peso a ciascuna parola in base alla sua frequenza nel documento e alla sua rarità nel corpus.

⚠️ Il vectorizer viene addestrato **solo sul training set** per evitare data leakage.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)
print(f"Feature totali (parole/bigrammi): {X_train_vec.shape[1]}")

## 5. Addestramento dei modelli

Due modelli di **Regressione Logistica**: uno per la categoria, uno per la priorità.
Scelta per semplicità, interpretabilità e buone prestazioni su testi brevi.

In [ ]:
model_cat  = LogisticRegression(max_iter=1000)
model_prio = LogisticRegression(max_iter=1000)
model_cat.fit(X_train_vec, y_train_cat)
model_prio.fit(X_train_vec, y_train_prio)
print("Modelli addestrati con successo!")

## 6. Valutazione — Categoria

In [ ]:
pred_cat = model_cat.predict(X_test_vec)
print("=== RISULTATI — CATEGORIA ===")
print(f"Accuracy: {accuracy_score(y_test_cat, pred_cat):.2%}")
print(classification_report(y_test_cat, pred_cat))

## 7. Valutazione — Priorità

La priorità è più difficile da classificare perché è un concetto soggettivo e il dataset è sbilanciato (46 Alta vs 162 Bassa).

In [ ]:
pred_prio = model_prio.predict(X_test_vec)
print("=== RISULTATI — PRIORITÀ ===")
print(f"Accuracy: {accuracy_score(y_test_prio, pred_prio):.2%}")
print(classification_report(y_test_prio, pred_prio))

## 8. Matrice di Confusione

La **matrice di confusione** mostra quante previsioni corrette ed errate ha prodotto il modello per ciascuna classe.

- La **diagonale principale** (blu scuro) = previsioni corrette
- I valori **fuori diagonale** = errori di classificazione

Per la categoria ci aspettiamo una diagonale perfetta. Per la priorità, eventuali errori sulla classe Alta sono attesi a causa dello sbilanciamento del dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_cat = confusion_matrix(y_test_cat, pred_cat, labels=model_cat.classes_)
ConfusionMatrixDisplay(cm_cat, display_labels=model_cat.classes_).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Matrice di Confusione - Categoria")

cm_prio = confusion_matrix(y_test_prio, pred_prio, labels=model_prio.classes_)
ConfusionMatrixDisplay(cm_prio, display_labels=model_prio.classes_).plot(
    ax=axes[1], colorbar=False, cmap="Blues"
)
axes[1].set_title("Matrice di Confusione - Priorità")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Immagine salvata: confusion_matrices.png")

## 9. Salvataggio dei modelli

In [ ]:
pickle.dump(model_cat,  open("model_categoria.pkl", "wb"))
pickle.dump(model_prio, open("model_priorita.pkl",  "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl",       "wb"))
print("Salvati: model_categoria.pkl, model_priorita.pkl, vectorizer.pkl")

## 10. Salvataggio report metriche (JSON)

In [ ]:
report_cat  = classification_report(y_test_cat,  pred_cat,  output_dict=True)
report_prio = classification_report(y_test_prio, pred_prio, output_dict=True)

with open("report_categoria.json", "w", encoding="utf-8") as f:
    json.dump(report_cat, f, indent=4, ensure_ascii=False)
with open("report_priorita.json", "w", encoding="utf-8") as f:
    json.dump(report_prio, f, indent=4, ensure_ascii=False)

print(f"F1-score macro Categoria: {report_cat['macro avg']['f1-score']:.2%}")
print(f"F1-score macro Priorità:  {report_prio['macro avg']['f1-score']:.2%}")

## Riepilogo

Il training è completato. I modelli sono stati:
- ✅ Addestrati con Regressione Logistica su dati TF-IDF
- ✅ Valutati sul test set con Accuracy, Precision, Recall, F1-score
- ✅ Analizzati con la matrice di confusione
- ✅ Salvati in formato `.pkl` per la dashboard Streamlit

Per avviare la dashboard:
```bash
streamlit run app.py
```